# **Evaluacion del universo de inversion**

## **Resumen de este notebook**

Este notebook define el universo de inversión final que se utilizará para la optimización de la cartera.

A partir de la evaluación de la calidad de los datos realizada en el cuaderno anterior, se establecen criterios objetivos de selección de activos que se aplican para construir tanto el universo de inversión principal como un universo alternativo para el análisis de sensibilidad.

### **Objetivo de negocio**

Construir un universo de inversión sólido y diversificado que represente adecuadamente las oportunidades de inversión disponibles, garantizando al mismo tiempo una estimación estadística fiable para la optimización de la cartera.

### **Objetivo analitico**

Aplicar criterios de selección predefinidos para determinar qué activos se incluirán en el universo de inversión principal y defina un universo alternativo para el análisis de sensibilidad.

### **Preguntas de negocio a responder en este notebook**

1. ¿Qué activos cumplen con los criterios de inclusión predefinidos?

2. ¿Qué activos deben incluirse en el universo de inversión principal?

3. ¿Qué activos solo se considerarán durante el análisis de sensibilidad?

4. ¿Cómo se distribuye el universo de inversión final entre sectores y clases de activos?

5. ¿El universo seleccionado está suficientemente diversificado para la optimización de la cartera?

### **Entregables de este notebook**


- Universo de inversión principal.

- Universo de análisis de sensibilidad.

- Resumen de selección de activos.

- Lista final de activos para la optimización de la cartera.

### **Importacion de Librerias**

In [1]:
# Standard Library
from pathlib import Path
import sys

# Third-Party Libraries
import pandas as pd

### **Configuracion del proyecto**

In [2]:
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [3]:
from src.data_loader import (
    load_dataset,
    save_dataset,
)

In [4]:
RAW_DATA_PATH = "../data/raw/"
PROCESSED_DATA_PATH = "../data/processed/"

### **Carga de datos**

In [14]:
close_prices = load_dataset(
    PROCESSED_DATA_PATH + "close_prices.parquet"
)

data_quality = load_dataset(
    PROCESSED_DATA_PATH +"data_quality.parquet"
)

investment_universe = pd.read_csv(
    RAW_DATA_PATH + "investment_universe.csv"
)

### **Descripcion general de los datos**

In [7]:
close_prices.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1258 entries, 2021-06-01 to 2026-05-29
Data columns (total 32 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    1255 non-null   float64
 1   AVGO    1255 non-null   float64
 2   BEP     1255 non-null   float64
 3   BRK-B   1255 non-null   float64
 4   CC=F    1257 non-null   float64
 5   CHWY    1255 non-null   float64
 6   CL=F    1257 non-null   float64
 7   COUR    1255 non-null   float64
 8   DE      1255 non-null   float64
 9   DUOL    1215 non-null   float64
 10  ETSY    1255 non-null   float64
 11  FDX     1255 non-null   float64
 12  GC=F    1257 non-null   float64
 13  GE      1255 non-null   float64
 14  GEV     545 non-null    float64
 15  GOOGL   1255 non-null   float64
 16  GXO     1219 non-null   float64
 17  IEF     1255 non-null   float64
 18  JBHT    1255 non-null   float64
 19  JPM     1255 non-null   float64
 20  KC=F    1258 non-null   float64
 21  KO      1255 non-null   float6

In [15]:
data_quality.info()

<class 'pandas.DataFrame'>
Index: 32 entries, AAPL to W
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   first_date          32 non-null     datetime64[ms]
 1   last_date           32 non-null     datetime64[ms]
 2   observations        32 non-null     int64         
 3   missing_percentage  32 non-null     float64       
dtypes: datetime64[ms](2), float64(1), int64(1)
memory usage: 1.4 KB


In [17]:
data_quality

,first_date,last_date,observations,missing_percentage
Ticker,,,,
AAPL,2021-06-01,2026-05-29,1255,0.238474
AVGO,2021-06-01,2026-05-29,1255,0.238474
BEP,2021-06-01,2026-05-29,1255,0.238474
BRK-B,2021-06-01,2026-05-29,1255,0.238474
CC=F,2021-06-01,2026-05-29,1257,0.079491
CHWY,2021-06-01,2026-05-29,1255,0.238474
CL=F,2021-06-01,2026-05-29,1257,0.079491
COUR,2021-06-01,2026-05-29,1255,0.238474
DE,2021-06-01,2026-05-29,1255,0.238474


In [11]:
investment_universe.info()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ticker       32 non-null     str  
 1   asset_name   32 non-null     str  
 2   asset_class  32 non-null     str  
 3   sector       32 non-null     str  
dtypes: str(4)
memory usage: 2.5 KB


### **Definir los criterios de inclusión de activos**

Para garantizar que la optimización de la cartera se base en datos financieros fiables y representativos, cada activo debe cumplir un conjunto predefinido de criterios de elegibilidad.

Estos criterios buscan asegurar que las estimaciones de riesgo, rentabilidad y correlación se calculen utilizando información histórica suficiente, manteniendo al mismo tiempo un universo de inversión diversificado.

Los siguientes criterios se evaluarán antes de construir el universo de inversión final.

In [20]:
selection_policy = pd.DataFrame(
    {
        "Criterion": [
            "Historical Coverage",
            "Missing Percentage",
        ],
        "Threshold": [
            "≥ 90%",
            "≤ 10%",
        ],
        "Business Rationale": [
            "Ensure reliable estimation of returns, volatility and correlations.",
            "Reduce the impact of incomplete time series on portfolio optimization.",
        ],
    }
)

selection_policy

,Criterion,Threshold,Business Rationale
0,Historical Coverage,≥ 90%,"Ensure reliable estimation of returns, volatil..."
1,Missing Percentage,≤ 10%,Reduce the impact of incomplete time series on...


In [21]:
# Asset selection criteria

# Minimum percentage of historical observations required
MIN_HISTORICAL_COVERAGE = 90  # 90%

# Maximum percentage of missing observations allowed
MAX_MISSING_PERCENTAGE = 10   # 10%

In [ ]:
# Add historical coverage 

data_quality["historical_coverage"] = (
    100 - data_quality["missing_percentage"]
)

In [ ]:
# Pass/Fall Flags

data_quality["passes_coverage"] = (
    data_quality["historical_coverage"] >= MIN_HISTORICAL_COVERAGE
)

data_quality["passes_missing"] = (
    data_quality["missing_percentage"] <= MAX_MISSING_PERCENTAGE
)

In [ ]:
# Eligibility

data_quality["eligible"] = (
    data_quality["passes_coverage"]
    &
    data_quality["passes_missing"]
)

In [25]:
data_quality

,first_date,last_date,observations,missing_percentage,historical_coverage,passes_coverage,passes_missing,eligible
Ticker,,,,,,,,
AAPL,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True
AVGO,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True
BEP,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True
BRK-B,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True
CC=F,2021-06-01,2026-05-29,1257,0.079491,99.920509,True,True,True
CHWY,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True
CL=F,2021-06-01,2026-05-29,1257,0.079491,99.920509,True,True,True
COUR,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True
DE,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True


In [26]:
from src.portfolio import get_selection_reason

In [ ]:
# Selection reason

data_quality["selection_reason"] = data_quality.apply(
    get_selection_reason,
    axis=1
)

In [28]:
data_quality

,first_date,last_date,observations,missing_percentage,historical_coverage,passes_coverage,passes_missing,eligible,selection_reason
Ticker,,,,,,,,,
AAPL,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True,Eligible
AVGO,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True,Eligible
BEP,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True,Eligible
BRK-B,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True,Eligible
CC=F,2021-06-01,2026-05-29,1257,0.079491,99.920509,True,True,True,Eligible
CHWY,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True,Eligible
CL=F,2021-06-01,2026-05-29,1257,0.079491,99.920509,True,True,True,Eligible
COUR,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True,Eligible
DE,2021-06-01,2026-05-29,1255,0.238474,99.761526,True,True,True,Eligible


### **Evaluar el universo de inversion**

In [29]:
data_quality["eligible"].value_counts().rename_axis("Eligible").reset_index(name="Count")

,Eligible,Count
0,True,31
1,False,1


In [30]:
data_quality.loc[
    ~data_quality["eligible"]
]

,first_date,last_date,observations,missing_percentage,historical_coverage,passes_coverage,passes_missing,eligible,selection_reason
Ticker,,,,,,,,,
GEV,2024-03-27,2026-05-29,545,56.677266,43.322734,False,False,False,Insufficient historical coverage and excessive...



La aplicación de la política de selección predefinida resultó en 27 activos elegibles y 1 activo excluido.

El activo excluido (GEV) no cumplió con el requisito mínimo de cobertura histórica y superó el porcentaje máximo permitido de observaciones faltantes. Esto concuerda con su historial de cotización reciente, que proporciona datos insuficientes para una estimación sólida de los rendimientos esperados, la volatilidad y las correlaciones.

Por consiguiente, GEV se excluirá del universo de inversión principal y se evaluará por separado en un análisis de sensibilidad.

### **Contruir el universo de inversion principal**

In [31]:
eligible_tickers = data_quality.loc[
    data_quality["eligible"]
].index

primary_universe = investment_universe[
    investment_universe["ticker"].isin(eligible_tickers)
].copy()

primary_universe.reset_index(drop=True, inplace=True)

### **Construir el analisis de sensibilidad del universo de inversion**

In [32]:
additional_assets = investment_universe[
    investment_universe["ticker"].isin(["GEV"])
]

sensitivity_universe = pd.concat(
    [primary_universe, additional_assets],
    ignore_index=True,
).drop_duplicates(
    subset="ticker"
)

### **Conclusiones**

El universo de inversión se evaluó utilizando criterios de calidad de datos predefinidos para garantizar que todos los activos incluidos en la optimización de la cartera proporcionaran suficiente información histórica.

La aplicación de la política de selección resultó en 31 activos elegibles y 1 excluido.

GEV no cumplió con el requisito mínimo de cobertura histórica debido a su reciente historial de cotización. En lugar de excluir permanentemente el activo del proyecto, se incorporará a un análisis de sensibilidad independiente para evaluar su impacto potencial en la construcción de la cartera.

Por lo tanto, se definieron dos universos de inversión:

- Universo de Inversión Primario: activos que cumplen con todos los criterios de selección.

- Universo de Inversión de Sensibilidad: universo primario más GEV.

Estos conjuntos de datos servirán de base para el proceso de optimización de la cartera desarrollado en el siguiente cuaderno.

### **Exportacion de entregables**

In [33]:
from src.data_loader import save_dataset

save_dataset(
    primary_universe,
    PROCESSED_DATA_PATH + "primary_universe.parquet"
)

save_dataset(
    sensitivity_universe,
    PROCESSED_DATA_PATH + "sensitivity_universe.parquet"
)

In [34]:
print(f"Primary Universe: {len(primary_universe)} assets")
print(f"Sensitivity Universe: {len(sensitivity_universe)} assets")

Primary Universe: 31 assets
Sensitivity Universe: 32 assets
